In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerIntegratedGradients, visualization as viz

# 日本語フォント設定
plt.rcParams['font.family'] = 'Meiryo'
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 設定: モデルと対象ジャンル
BASE_MODEL_NAME = "tohoku-nlp/bert-large-japanese-v2"
RESULT_DIR_PREFIX = "result_"
RESULT_DIR_PATH = "result_all/checkpoint-300"

# 分析対象のジャンルID (ここで指定したジャンルに対応するcheckpointが下から選ばれます)
TARGET_GENRE_ID = 101

# 設定: ジャンルごとのチェックポイント対応表
# ここに手動で、各ジャンルフォルダ内にある「最大の数値のcheckpoint名」を記述してください
genre_checkpoint_map = {
    0:    "checkpoint-3000", # 未設定
    101:  "checkpoint-300", # 異世界（恋愛）
    102:  "checkpoint-300", # 現実世界（恋愛）
    201:  "checkpoint-3000", # ハイファンタジー
    202:  "checkpoint-3000", # ローファンタジー
    301:  "checkpoint-750", # 純文学
    302:  "checkpoint-3855", # ヒューマンドラマ
    303:  "checkpoint-750", # 歴史
    304:  "checkpoint-1050", # 推理
    305:  "checkpoint-1200", # ホラー
    306:  "checkpoint-900", # アクション
    307:  "checkpoint-3000", # コメディー
    401:  "checkpoint-3000", # VRゲーム
    402:  "checkpoint-300", # 宇宙
    403:  "checkpoint-3000", # 空想科学
    404:  "checkpoint-3000", # パニック
    9901: "checkpoint-300", # 童話
    9902: "checkpoint-3000", # 詩
    9903: "checkpoint-3000", # エッセイ
    9904: "checkpoint-3000", # リプレイ
    9999: "checkpoint-3000", # その他
    9801: "checkpoint-6051", # ノンジャンル
}
# ==========================================

# ジャンルマップ
genres_map = {
    0: '未設定', 101: '異世界（恋愛）', 102: '現実世界（恋愛）', 201: 'ハイファンタジー', 202: 'ローファンタジー',
    301: '純文学', 302: 'ヒューマンドラマ', 303: '歴史', 304: '推理', 305: 'ホラー',
    306: 'アクション', 307: 'コメディー', 401: 'VRゲーム', 402: '宇宙', 403: '空想科学',
    404: 'パニック', 9901: '童話', 9902: '詩', 9903: 'エッセイ', 9904: 'リプレイ',
    9999: 'その他', 9801: 'ノンジャンル'
}

print("設定完了")

設定完了


In [ ]:
# --- モデルの読み込み ---
target_genre_name = genres_map.get(TARGET_GENRE_ID, '不明')
print(f"選択されたジャンル: {target_genre_name} ({TARGET_GENRE_ID})")

# 対応するチェックポイント名を取得
checkpoint_name = genre_checkpoint_map.get(TARGET_GENRE_ID)

if checkpoint_name is None:
    print(f"【エラー】ジャンルID {TARGET_GENRE_ID} に対応するチェックポイントが設定されていません。")
    print("セル1の genre_checkpoint_map を確認してください。")
    model = None
else:
    # パスの構築
    model_dir = f"{RESULT_DIR_PREFIX}{target_genre_name}/{checkpoint_name}"
    #model_dir = RESULT_DIR_PATH
    print(f"モデルを読み込んでいます... Path: {model_dir}")

    if os.path.exists(model_dir):
        # GPU設定
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"使用デバイス: {device}")
        
        # モデルとトークナイザ
        try:
            model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
            model.eval() # 評価モード
            model.zero_grad() # 勾配リセット

            print(f"トークナイザを {BASE_MODEL_NAME} からロードします...")
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
            
            print("モデル読み込み完了。")
        except Exception as e:
            print(f"【エラー】モデル読み込み失敗: {e}")
            model = None
    else:
        print(f"【エラー】モデルが見つかりません: {model_dir}")
        print("フォルダ名やチェックポイント名が正しいか確認してください。")
        model = None

選択されたジャンル: 異世界（恋愛） (101)
モデルを読み込んでいます... Path: result_異世界（恋愛）/checkpoint-300
使用デバイス: cuda
トークナイザを tohoku-nlp/bert-large-japanese-v2 からロードします...
モデル読み込み完了。


In [11]:
# --- データの読み込みと抽出 ---
corpus_path = 'dataset/narou_dataset.csv'

try:
    df = pd.read_csv(corpus_path)
    # あらすじ
    df['text'] = df['あらすじ'].fillna('')
    
    # 指定したジャンルからランダム抽出
    df_genre = df[df['作品ジャンル'] == TARGET_GENRE_ID]
    
    if len(df_genre) > 0:
        random_sample = df_genre.sample(n=1)
        
        target_text = random_sample['text'].values[0]
        target_label = random_sample['is_eternal'].values[0]
        target_title = random_sample['タイトル'].values[0]

        print(f"\n=== 分析対象 (ランダム抽出) ===")
        print(f"タイトル: {target_title}")
        print(f"ジャンル: {TARGET_GENRE_ID} ({target_genre_name})")
        print(f"正解: {'エタる (1)' if target_label == 1 else '完結 (0)'}")
        print("-" * 30)
    else:
        print(f"エラー: ジャンルID {TARGET_GENRE_ID} のデータがありません。")
        target_text = None

except Exception as e:
    print(f"データ読み込みエラー: {e}")
    target_text = None


=== 分析対象 (ランダム抽出) ===
タイトル: 猫耳系下っ端悪役令嬢に転生しました！
ジャンル: 101 (異世界（恋愛）)
正解: 完結 (0)
------------------------------


In [12]:
# --- セル4: Captum (Integrated Gradients) による分析実行 ---

if model is not None and target_text is not None:
    print("\nCaptumによる分析を開始します...")

    # --- 1. BERTの予測関数ラッパー ---
    # Captumから「input_ids」と「attention_mask」を受け取って、モデルの出力(logits)を返す関数
    def predict_func(input_ids, attention_mask=None):
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits

    # Integrated Gradientsの準備
    # model.bert.embeddings をターゲットにして、入力（単語）の貢献度を計算します
    lig = LayerIntegratedGradients(predict_func, model.bert.embeddings)

    # --- 2. データの準備 ---
    # 入力テキストをID化
    inputs = tokenizer(
        target_text, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to(device)
    
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    # --- 3. 予測の実行 ---
    with torch.no_grad():
        logits = model(**inputs).logits
        pred_label_idx = torch.argmax(logits, dim=1).item()
        probs = torch.nn.functional.softmax(logits, dim=1)[0]

    # ★追加: 「隠れた初期値（ベースラインのスコア）」を確認する
    # 何も入力がない時にAIがどう判定しているかを見ることで、Attribution Scoreがマイナスになる理由がわかります
    with torch.no_grad():
        baseline_inputs = torch.zeros_like(input_ids).to(device)
        baseline_logits = model(input_ids=baseline_inputs, attention_mask=attention_mask).logits
        
    print(f"【AI予測】: {'エタる (1)' if pred_label_idx==1 else 'エタらない (0)'} (確率: {probs[pred_label_idx]:.4f})")
    print(f"※参考: 空っぽの時のスコア(Logits): {baseline_logits[0].cpu().numpy()} (この値が高いと、単語スコアがマイナスでもエタる判定になります)")

    # --- 4. 貢献度（Attributions）の計算 ---
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=torch.zeros_like(input_ids), # ベースライン（[PAD]トークン）
        target=pred_label_idx,
        # ★重要: カンマをつけてタプルとして渡す
        additional_forward_args=(attention_mask, ), 
        return_convergence_delta=True,
        internal_batch_size=4 
    )

    # --- 5. 結果の集計 ---
    attributions = attributions.sum(dim=2).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    attributions = attributions.cpu().detach().numpy()

    # トークンのリストを取得
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # --- 6. 結果の可視化 ---
    print(f"\n【重要度可視化】")
    print("赤色: 予測ラベル（エタる/エタらない）を「支持」した単語 (プラスの貢献)")
    print("青色: 予測ラベルを「否定」しようとした単語 (マイナスの貢献)")
    print("※ [UNK]や##がついた単語は、AIが内部で文字を分解して理解している様子です")

    score_vis = viz.VisualizationDataRecord(
        word_attributions=attributions,
        pred_prob=probs[pred_label_idx],
        pred_class=pred_label_idx,
        true_class=target_label,
        attr_class=str(pred_label_idx),
        attr_score=attributions.sum(),
        raw_input_ids=tokens,
        convergence_score=delta
    )
    
    viz.visualize_text([score_vis])

else:
    print("モデルまたはデータが準備できていません。セル1〜3を確認してください。")


Captumによる分析を開始します...
【AI予測】: エタらない (0) (確率: 0.8718)
※参考: 空っぽの時のスコア(Logits): [ 0.9200184 -1.2873156] (この値が高いと、単語スコアがマイナスでもエタる判定になります)

【重要度可視化】
赤色: 予測ラベル（エタる/エタらない）を「支持」した単語 (プラスの貢献)
青色: 予測ラベルを「否定」しようとした単語 (マイナスの貢献)
※ [UNK]や##がついた単語は、AIが内部で文字を分解して理解している様子です
